# Movement notebook

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('/root/movement')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
from data_prep.setup import init_device_and_seeds
device = init_device_and_seeds(42)

In [ ]:
from transformers import AutoProcessor, VitPoseForPoseEstimation

MODEL_REPO = "usyd-community/vitpose-plus-large"
image_processor = AutoProcessor.from_pretrained(MODEL_REPO)
model = VitPoseForPoseEstimation.from_pretrained(MODEL_REPO)
model.to(device)
model.eval()
print("VitPose loaded on", device)


In [ ]:
from pathlib import Path
from data_prep.paths import prepare_sample_paths
DATA_DIR = Path("/root/movement/data/kinetics-dataset/k700-2020/train")
SAMPLES = [
    "acting in play/0bdVrgImymc_000020_000030",
    "acting in play/0bk_zXtqOTQ_000020_000030",
    "acting in play/0lZhBK_QOak_000283_000293",
]
filename = SAMPLES[2]; person_id = "1"
raw_video_path, tracking_path = prepare_sample_paths(DATA_DIR, filename)
print("Data dir:", DATA_DIR.resolve()); print("Selected:", filename)
print("Raw candidate:", raw_video_path)


In [ ]:
# Local-only check for Kinetics raw file
raw_candidate = (DATA_DIR / f"{filename}.mp4").resolve()
raw_video_path = str(raw_candidate)
print("Raw candidate:", raw_candidate, "exists:", raw_candidate.exists())

In [ ]:
from data_prep.tracking import run_tracking_and_save
tracking_path = (DATA_DIR / f"{filename}.json").resolve()
print("Writing tracking to:", tracking_path)
frame_idxs, boxes_xyxy = run_tracking_and_save(raw_video_path, tracking_path, device, person_id)
print(f"Saved {len(frame_idxs)} tracked frames for person_id={person_id}")


In [ ]:
import json
import cv2
import numpy as np
import torch
import PIL

# Load tracking JSON and use first tracked frame/box
tracking_path = (DATA_DIR / f"{filename}.json").resolve()
with open(tracking_path, "r") as f:
    tracking = json.load(f)

idxs = tracking[str(person_id)]["frame_idxs"]
boxes_xyxy = tracking[str(person_id)]["box"]
if not idxs:
    raise RuntimeError("Tracking JSON has no frames")

target_idx = int(idxs[0])
xyxy = boxes_xyxy[0]  # [x1,y1,x2,y2]
# convert to COCO [x,y,w,h] as (1,4) float32
person_boxes_coco = np.expand_dims(np.array([xyxy[0], xyxy[1], xyxy[2]-xyxy[0], xyxy[3]-xyxy[1]], dtype=np.float32), axis=0)

# Seek and read that frame
cap = cv2.VideoCapture(raw_video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"Failed to read frame {target_idx}")

h, w = frame_bgr.shape[:2]
rgb_frame = frame_bgr[:, :, ::-1]
image = PIL.Image.fromarray(rgb_frame)

# Prepare inputs for ViTPose with box (exact pattern from metabot_gendata)
pixel_values = image_processor(image, boxes=[person_boxes_coco], return_tensors="pt").pixel_values

# dataset_index as 1D tensor on device
dataset_index = torch.tensor([0], device=device)

pixel_values = pixel_values.to(device=device)
with torch.no_grad():
    outputs = model(pixel_values, dataset_index=dataset_index)

pose_results = image_processor.post_process_pose_estimation(outputs, boxes=[person_boxes_coco])
image_pose_result = pose_results[0][0]  # results for first image

xy = np.expand_dims(image_pose_result["keypoints"].cpu().numpy(), axis=0)
scores = np.expand_dims(image_pose_result["scores"].cpu().numpy(), axis=0)

In [ ]:
from PIL import Image

assert 'pixel_values' in locals(), "pixel_values not available; run the pose cell first."

batch_index = 0
mean = image_processor.image_mean
std = image_processor.image_std
img_np = pixel_values[batch_index].detach().cpu().numpy()
unnormalized = (img_np * np.array(std)[:, None, None]) + np.array(mean)[:, None, None]
unnormalized = (unnormalized * 255).astype(np.uint8)
unnormalized = np.moveaxis(unnormalized, 0, -1)
Image.fromarray(unnormalized)

In [ ]:
assert 'outputs' in locals(), "outputs not available; run the pose cell first."
assert hasattr(outputs, 'heatmaps'), "This model checkpoint does not expose raw heatmaps."

from data_prep.visualization import visualize_layered_heatmaps

visualize_layered_heatmaps(outputs.heatmaps.detach().cpu().numpy())

In [ ]:
from data_prep.visualization import overlay_pose_and_bbox
import numpy as np
import matplotlib.pyplot as plt

assert 'xy' in locals() and xy is not None, "xy not available; run the pose cell first."
assert 'scores' in locals() and scores is not None, "scores not available; run the pose cell first."
assert 'rgb_frame' in locals(), "rgb_frame not available; run the pose cell first."
assert 'xyxy' in locals(), "xyxy (bbox) not available; run the pose cell first."

annot_img = rgb_frame.copy()

# Build KeyPoints object as in metabot_gendata
keypoints = sv.KeyPoints(xy=xy.astype(np.float32), confidence=scores.astype(np.float32))
# Build Detections for optional bbox drawing (expects xyxy)
detections = sv.Detections(xyxy=np.expand_dims(np.array(xyxy, dtype=np.float32), axis=0))

edge_annotator = sv.EdgeAnnotator(color=sv.Color.GREEN, thickness=3)
vertex_annotator = sv.VertexAnnotator(color=sv.Color.RED, radius=2)
bounding_box_annotator = sv.BoxAnnotator(color=sv.Color.BLUE, color_lookup=sv.ColorLookup.INDEX, thickness=1)

# Annotate
annot_img = edge_annotator.annotate(scene=annot_img, key_points=keypoints)
annot_img = vertex_annotator.annotate(scene=annot_img, key_points=keypoints)
annot_img = bounding_box_annotator.annotate(scene=annot_img, detections=detections)

plt.figure(figsize=(6,6))
plt.imshow(annot_img)
plt.axis('off')
plt.title('ViTPose keypoints')
plt.show()


In [ ]:
from data_prep.visualization import overlay_pose_and_bbox
import matplotlib.pyplot as plt

annot_img = overlay_pose_and_bbox(rgb_frame, xy, scores, xyxy)
plt.figure(figsize=(6,6))
plt.imshow(annot_img)
plt.axis('off')
plt.title('ViTPose keypoints (modular)')
plt.show()



In [ ]:
from data_prep.pose3d import build_motionagformer
from MotionAGFormer.model.MotionAGFormer import MotionAGFormer
ckpt_path = Path('/root/movement/models/motionagformer-b-h36m.pth.tr')
assert ckpt_path.exists(), f'Checkpoint not found at {ckpt_path}'
model_3d = build_motionagformer(MotionAGFormer, device, ckpt_path)
print('MotionAGFormer loaded from', ckpt_path)


In [ ]:
from data_prep.visualization import probe_video_size
height, width = probe_video_size(raw_video_path)
print("Video size:", width, "x", height)


In [ ]:
from data_prep.constants import CLIP_LEN, CLIP_HOP
import importlib
import data_prep.pose3d as _pose3d
importlib.reload(_pose3d)
from data_prep.pose3d import lift_sequence_to_3d

use_overlap = True

y3d = lift_sequence_to_3d(
    seq_keypoints=seq_keypoints,
    seq_scores=seq_scores,
    width=width,
    height=height,
    model_3d=model_3d,
    device=device,
    use_overlap=use_overlap,
    clip_len=CLIP_LEN,
    clip_hop=CLIP_HOP,
)

In [ ]:
import matplotlib.pyplot as plt

from data_prep.constants import H36M_I as I, H36M_J as J

from data_prep.visualization import plot_pose_equal_axes

# usage on your first frame of the first clip:
plot_pose_equal_axes(all_3d[0][0], title="3D Pose (clip 0, frame 0)")

In [ ]:
from data_prep.visualization import draw_h36m_on_video_frame
img = draw_h36m_on_video_frame(raw_video_path, idxs, seq_keypoints, seq_scores, seq_idx=0)
plt.figure(figsize=(7,6))
plt.imshow(img)
plt.title(f'H36M overlay — raw frame {int(idxs[0])} (seq idx 0)')
plt.axis('off')
plt.show()


In [ ]:
from data_prep.geometry import fit_similarity_2d
from data_prep.visualization import overlay_3d_on_video_inline

stride = 2
resize_scale = 0.5
max_fps = 20.0

overlay_3d_on_video_inline(
    all_3d=all_3d,
    seq_keypoints=seq_keypoints,
    idxs=idxs,
    raw_video_path=raw_video_path,
    fit_similarity_2d_fn=fit_similarity_2d,
    stride=stride,
    resize_scale=resize_scale,
    max_fps=max_fps,
)